# Version 22.1 : Triple Ensemble avec Pipelines Complets

**Stratégie** :
- 🚀 **XGBoost 6.1** : Training direct (comme V22)
- 📊 **CoxNet 21.1** : Cross-validation complète pour alpha/l1_ratio (comme V21.1)
- 🧠 **DeepSurv 20.1** : Training complet avec early stopping (comme V20.1)
- ⚡ **Ensemble** : Optimisation Optuna des poids

**Objectif** : C-index > **0.755+**

## 1. Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import optuna
import os

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import concordance_index_censored
from sksurv.util import Surv

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

print("✓ Libraries imported")

✓ Libraries imported


In [2]:
DATA_PATH = "C:/Users/guill/Desktop/Data Challenge QRT/Data-Challenge-Prediction-de-Survie"

clinical_train = pd.read_csv(os.path.join(DATA_PATH, "X_train", "clinical_train.csv"))
target_train = pd.read_csv(os.path.join(DATA_PATH, "target_train.csv"))
clinical_test = pd.read_csv(os.path.join(DATA_PATH, "X_test", "clinical_test.csv"))
molecular_train = pd.read_csv(os.path.join(DATA_PATH, "X_train", "molecular_train.csv"))
molecular_test = pd.read_csv(os.path.join(DATA_PATH, "X_test", "molecular_test.csv"))

print(f"✓ Data loaded: {clinical_train.shape[0]} train patients")

✓ Data loaded: 3323 train patients


## 2. Feature Engineering (Unified Enriched Features)

In [3]:
def create_cytogenetic_features(clinical_df):
    cyto_features = pd.DataFrame(index=clinical_df['ID'])
    cyto_col = clinical_df.set_index('ID')['CYTOGENETICS'].fillna('')
    
    cyto_features['cyto_del_count'] = cyto_col.str.count(r'del\(')
    cyto_features['cyto_has_del'] = (cyto_features['cyto_del_count'] > 0).astype(int)
    cyto_features['cyto_transloc_count'] = cyto_col.str.count(r't\(')
    cyto_features['cyto_has_transloc'] = (cyto_features['cyto_transloc_count'] > 0).astype(int)
    cyto_features['cyto_inv_count'] = cyto_col.str.count(r'inv\(')
    cyto_features['cyto_has_inv'] = (cyto_features['cyto_inv_count'] > 0).astype(int)
    cyto_features['cyto_gain_count'] = cyto_col.str.count(r'\+')
    cyto_features['cyto_has_gain'] = (cyto_features['cyto_gain_count'] > 0).astype(int)
    cyto_features['cyto_loss_count'] = cyto_col.str.count(r'-[0-9XY]')
    cyto_features['cyto_has_loss'] = (cyto_features['cyto_loss_count'] > 0).astype(int)
    cyto_features['cyto_other_count'] = cyto_col.str.count(r'add\(|ins\(|dup\(')
    
    cyto_features['cyto_total_anomalies'] = (
        cyto_features['cyto_del_count'] + cyto_features['cyto_transloc_count'] + 
        cyto_features['cyto_inv_count'] + cyto_features['cyto_gain_count'] + 
        cyto_features['cyto_loss_count'] + cyto_features['cyto_other_count']
    )
    cyto_features['cyto_normal'] = cyto_col.str.match(r'^46,(xx|xy)(\[\d+\])?$', case=False).astype(int)
    cyto_features['cyto_complex'] = (
        (cyto_features['cyto_total_anomalies'] >= 3) | 
        cyto_col.str.contains('complex', case=False, na=False)
    ).astype(int)
    
    chromosomes = [str(i) for i in range(1, 23)] + ['X', 'Y']
    for chrom in chromosomes:
        pattern = rf'(\b|[,\(]){chrom}([,;:\)\[]|[pq])'
        cyto_features[f'cyto_chr{chrom}_affected'] = cyto_col.str.contains(
            pattern, case=False, na=False, regex=True
        ).astype(int)
    
    cyto_features['cyto_monosomy7'] = cyto_col.str.contains(r'-7[^0-9]|^45.*-7', case=False, na=False, regex=True).astype(int)
    cyto_features['cyto_trisomy8'] = cyto_col.str.contains(r'\+8[^0-9]|^47.*\+8', case=False, na=False, regex=True).astype(int)
    cyto_features['cyto_del5q'] = cyto_col.str.contains(r'del\(5\)\(q', case=False, na=False, regex=True).astype(int)
    cyto_features['cyto_del20q'] = cyto_col.str.contains(r'del\(20\)\(q', case=False, na=False, regex=True).astype(int)
    cyto_features['cyto_chr3_abnormal'] = cyto_col.str.contains(r'(del|t|inv)\(3[;,:\)]', case=False, na=False, regex=True).astype(int)
    cyto_features['cyto_chr7_abnormal'] = cyto_col.str.contains(r'(del|t|inv)\(7[;,:\)]', case=False, na=False, regex=True).astype(int)
    
    return cyto_features.fillna(0)

def create_molecular_features(molecular_df, patient_ids, top_n_genes=20):
    mol_features = pd.DataFrame({'ID': patient_ids})
    mutation_counts = molecular_df.groupby('ID').size().to_frame('mutation_count_total')
    mol_features = mol_features.merge(mutation_counts, on='ID', how='left')
    
    vaf_stats = molecular_df.groupby('ID')['VAF'].agg([
        ('vaf_mean', 'mean'), ('vaf_max', 'max'), ('vaf_sum', 'sum')
    ]).reset_index()
    mol_features = mol_features.merge(vaf_stats, on='ID', how='left')
    
    effect_counts = molecular_df.groupby(['ID', 'EFFECT']).size().unstack(fill_value=0)
    effect_counts.columns = [f'effect_{col}' for col in effect_counts.columns]
    mol_features = mol_features.merge(effect_counts.reset_index(), on='ID', how='left')
    
    top_genes_list = molecular_df['GENE'].value_counts().head(top_n_genes).index.tolist()
    for gene in top_genes_list:
        gene_mutations = molecular_df[molecular_df['GENE'] == gene].groupby('ID').size()
        mol_features[f'gene_{gene}_count'] = mol_features['ID'].map(gene_mutations)
        gene_present = molecular_df[molecular_df['GENE'] == gene]['ID'].unique()
        mol_features[f'gene_{gene}_present'] = mol_features['ID'].isin(gene_present).astype(int)
    
    feature_cols = [col for col in mol_features.columns if col != 'ID']
    mol_features[feature_cols] = mol_features[feature_cols].fillna(0)
    return mol_features.set_index('ID')

def add_enriched_features(X_clinical, X_molecular):
    X_enriched = pd.DataFrame(index=X_clinical.index)
    
    for col in ['BM_BLAST', 'WBC', 'ANC', 'MONOCYTES']:
        if col in X_clinical.columns:
            X_enriched[f'{col}_log'] = np.log1p(X_clinical[col])
            X_enriched[f'{col}_sqrt'] = np.sqrt(X_clinical[col].clip(lower=0))
            X_enriched[f'{col}_squared'] = X_clinical[col] ** 2
    
    if 'WBC' in X_clinical.columns and 'ANC' in X_clinical.columns:
        X_enriched['WBC_ANC_ratio'] = X_clinical['WBC'] / (X_clinical['ANC'] + 1e-5)
    if 'HB' in X_clinical.columns and 'PLT' in X_clinical.columns:
        X_enriched['HB_PLT_ratio'] = X_clinical['HB'] / (X_clinical['PLT'] + 1e-5)
    if 'WBC' in X_clinical.columns and 'MONOCYTES' in X_clinical.columns:
        X_enriched['WBC_MONOCYTES_ratio'] = X_clinical['WBC'] / (X_clinical['MONOCYTES'] + 1e-5)
    if 'BM_BLAST' in X_clinical.columns and 'WBC' in X_clinical.columns:
        X_enriched['BLAST_WBC_ratio'] = X_clinical['BM_BLAST'] / (X_clinical['WBC'] + 1e-5)
    
    important_pairs = [
        ('BM_BLAST', 'WBC'), ('BM_BLAST', 'HB'), ('BM_BLAST', 'PLT'),
        ('WBC', 'HB'), ('HB', 'PLT'), ('ANC', 'MONOCYTES')
    ]
    for col1, col2 in important_pairs:
        if col1 in X_clinical.columns and col2 in X_clinical.columns:
            X_enriched[f'{col1}_x_{col2}'] = X_clinical[col1] * X_clinical[col2]
    
    if 'mutation_count_total' in X_molecular.columns:
        if 'BM_BLAST' in X_clinical.columns:
            X_enriched['mutations_x_BLAST'] = X_molecular['mutation_count_total'] * X_clinical['BM_BLAST']
        if 'WBC' in X_clinical.columns:
            X_enriched['mutations_x_WBC'] = X_molecular['mutation_count_total'] * X_clinical['WBC']
    if 'vaf_sum' in X_molecular.columns and 'BM_BLAST' in X_clinical.columns:
        X_enriched['vaf_sum_x_BLAST'] = X_molecular['vaf_sum'] * X_clinical['BM_BLAST']
    
    if 'BM_BLAST' in X_clinical.columns:
        X_enriched['BLAST_low'] = (X_clinical['BM_BLAST'] < 20).astype(int)
        X_enriched['BLAST_high'] = (X_clinical['BM_BLAST'] >= 50).astype(int)
    
    return X_enriched

print("✓ Feature engineering functions defined")

✓ Feature engineering functions defined


In [4]:
# Build features
target_clean = target_train.dropna(subset=['OS_YEARS', 'OS_STATUS']).copy()
target_clean['OS_STATUS'] = target_clean['OS_STATUS'].astype(bool)
target_clean = target_clean.set_index('ID')

clinical_train_clean = clinical_train[clinical_train['ID'].isin(target_clean.index)].copy()
clinical_train_clean = clinical_train_clean.set_index('ID').loc[target_clean.index]

numeric_features = ['BM_BLAST', 'WBC', 'ANC', 'MONOCYTES', 'HB', 'PLT']
X_clinical_train = clinical_train_clean[numeric_features].copy()
center_encoded_train = pd.get_dummies(clinical_train_clean['CENTER'], prefix='CENTER', drop_first=True)
X_clinical_train = pd.concat([X_clinical_train, center_encoded_train], axis=1)

cyto_features_train = create_cytogenetic_features(clinical_train)
train_patient_ids = clinical_train_clean.index.unique()
mol_features_train = create_molecular_features(molecular_train, train_patient_ids, top_n_genes=20)
mol_features_train_aligned = mol_features_train.reindex(X_clinical_train.index, fill_value=0)

enriched_features_train = add_enriched_features(X_clinical_train[numeric_features], mol_features_train_aligned)

cyto_features_train_aligned = cyto_features_train.reindex(X_clinical_train.index, fill_value=0)
X_combined_train = pd.concat([
    X_clinical_train, mol_features_train_aligned, cyto_features_train_aligned, enriched_features_train
], axis=1)

# Test set
X_clinical_test = clinical_test.set_index('ID')[numeric_features].copy()
center_encoded_test = pd.get_dummies(clinical_test.set_index('ID')['CENTER'], prefix='CENTER', drop_first=True)
X_clinical_test = pd.concat([X_clinical_test, center_encoded_test], axis=1)

for col in X_clinical_train.columns:
    if col not in X_clinical_test.columns:
        X_clinical_test[col] = 0
X_clinical_test = X_clinical_test[X_clinical_train.columns]

cyto_features_test = create_cytogenetic_features(clinical_test)
test_patient_ids = clinical_test['ID'].unique()
mol_features_test = create_molecular_features(molecular_test, test_patient_ids, top_n_genes=20)

for col in mol_features_train.columns:
    if col not in mol_features_test.columns:
        mol_features_test[col] = 0
mol_features_test = mol_features_test[mol_features_train.columns]

mol_features_test_aligned = mol_features_test.reindex(X_clinical_test.index, fill_value=0)
enriched_features_test = add_enriched_features(X_clinical_test[numeric_features], mol_features_test_aligned)
cyto_features_test_aligned = cyto_features_test.reindex(X_clinical_test.index, fill_value=0)

X_combined_test = pd.concat([
    X_clinical_test, mol_features_test_aligned, cyto_features_test_aligned, enriched_features_test
], axis=1)

y_surv = Surv.from_dataframe('OS_STATUS', 'OS_YEARS', target_clean)

print(f"✓ Features created: {X_combined_train.shape}")

✓ Features created: (3173, 159)


## 3. Train/Val Split (10% val for ensemble weights only)

In [5]:
# Small validation set for ensemble weight optimization
X_train_all, X_val_ensemble, y_train_all, y_val_ensemble = train_test_split(
    X_combined_train, y_surv, test_size=0.1, random_state=42,
    stratify=target_clean['OS_STATUS'].astype(int)
)

print(f"✓ Split: {X_train_all.shape[0]} train (90%), {X_val_ensemble.shape[0]} val (10%)")

✓ Split: 2855 train (90%), 318 val (10%)


## 4. MODEL 1: XGBoost 6.1 (Direct Training)

In [6]:
print("="*80)
print("MODEL 1: XGBOOST 6.1")
print("="*80)

XGB_PARAMS = {
    'objective': 'survival:cox',
    'eval_metric': 'cox-nloglik',
    'tree_method': 'hist',
    'learning_rate': 0.0744,
    'max_depth': 3,
    'min_child_weight': 9,
    'subsample': 0.6194,
    'colsample_bytree': 0.8872,
    'reg_alpha': 0.0124,
    'reg_lambda': 0.0043,
    'gamma': 0.0600,
}

imputer_xgb = SimpleImputer(strategy='median')
X_train_xgb = pd.DataFrame(
    imputer_xgb.fit_transform(X_train_all),
    index=X_train_all.index,
    columns=X_train_all.columns
)
X_val_xgb = pd.DataFrame(
    imputer_xgb.transform(X_val_ensemble),
    index=X_val_ensemble.index,
    columns=X_val_ensemble.columns
)

y_train_xgb = y_train_all['OS_YEARS'].copy()
y_train_xgb[~y_train_all['OS_STATUS']] = -y_train_xgb[~y_train_all['OS_STATUS']]
y_val_xgb = y_val_ensemble['OS_YEARS'].copy()
y_val_xgb[~y_val_ensemble['OS_STATUS']] = -y_val_xgb[~y_val_ensemble['OS_STATUS']]

dtrain = xgb.DMatrix(X_train_xgb, label=y_train_xgb)
dval = xgb.DMatrix(X_val_xgb, label=y_val_xgb)

xgb_model = xgb.train(
    XGB_PARAMS,
    dtrain,
    num_boost_round=500,
    evals=[(dval, 'val')],
    early_stopping_rounds=50,
    verbose_eval=False
)

y_pred_xgb_val = xgb_model.predict(dval)
c_index_xgb = concordance_index_censored(
    y_val_ensemble['OS_STATUS'], y_val_ensemble['OS_YEARS'], y_pred_xgb_val
)[0]

print(f"✓ XGBoost trained")
print(f"  C-index: {c_index_xgb:.4f}")

MODEL 1: XGBOOST 6.1
✓ XGBoost trained
  C-index: 0.7504


## 5. MODEL 2: CoxNet 21.1 (Full Cross-Validation from V21.1)

In [17]:
print("\n" + "="*80)
print("MODEL 2: COXNET 21.1 - LOADING PRE-TRAINED MODEL")
print("="*80)

# ============================================================================
# Load pre-trained CoxNet model from V21.1
# ============================================================================
import pickle

model_path = os.path.join(DATA_PATH, 'coxnet_v21.1_model.pkl')

if not os.path.exists(model_path):
    raise FileNotFoundError(
        f"❌ CoxNet model not found at {model_path}\n"
        f"Please run notebook 21.1_CoxNet_Enriched.ipynb first and save the model."
    )

with open(model_path, 'rb') as f:
    coxnet_artifacts = pickle.load(f)

cox_model_final = coxnet_artifacts['model']
imputer_cox_pretrained = coxnet_artifacts['imputer']
scaler_cox_pretrained = coxnet_artifacts['scaler']
feature_columns_21_1 = coxnet_artifacts['feature_columns']
best_params = coxnet_artifacts['best_params']
pretrained_performance = coxnet_artifacts['performance']

print(f"\n✓ Loaded pre-trained CoxNet V21.1")
print(f"  Original C-index: {pretrained_performance['c_index_val']:.4f}")
print(f"  Features: {len(feature_columns_21_1)}")
print(f"  Best params: l1_ratio={best_params['l1_ratio']}, alpha={best_params['alpha']}")

# ============================================================================
# Build V21.1-compatible features (without chr1-chr22 features)
# ============================================================================
print(f"\n📝 Building features compatible with V21.1...")

# Training data (90%)
X_clinical_train_21 = pd.DataFrame(index=X_train_all.index)
for col in numeric_features:
    if col in X_train_all.columns:
        X_clinical_train_21[col] = X_train_all[col]

center_cols_train = [col for col in X_train_all.columns if col.startswith('CENTER_')]
if len(center_cols_train) > 0:
    X_clinical_train_21 = pd.concat([X_clinical_train_21, X_train_all[center_cols_train]], axis=1)

mol_feature_cols = [col for col in X_train_all.columns if any(prefix in col for prefix in 
                    ['mutation_', 'vaf_', 'effect_', 'gene_'])]
mol_features_train_21 = X_train_all[mol_feature_cols]

cyto_feature_cols_21 = [col for col in X_train_all.columns if col.startswith('cyto_') and 
                        not col.startswith('cyto_chr') and
                        col not in ['cyto_other_count', 'cyto_chr3_abnormal', 'cyto_chr7_abnormal']]
cyto_features_train_21 = X_train_all[cyto_feature_cols_21]

enriched_features_train_21 = add_enriched_features(X_clinical_train_21[numeric_features], mol_features_train_21)

X_combined_train_21 = pd.concat([
    X_clinical_train_21, mol_features_train_21, cyto_features_train_21, enriched_features_train_21
], axis=1)

# Validation data (10%)
X_clinical_val_21 = pd.DataFrame(index=X_val_ensemble.index)
for col in numeric_features:
    if col in X_val_ensemble.columns:
        X_clinical_val_21[col] = X_val_ensemble[col]

if len(center_cols_train) > 0:
    center_cols_val = [col for col in X_val_ensemble.columns if col.startswith('CENTER_')]
    X_clinical_val_21 = pd.concat([X_clinical_val_21, X_val_ensemble[center_cols_val]], axis=1)

mol_features_val_21 = X_val_ensemble[mol_feature_cols]
cyto_features_val_21 = X_val_ensemble[cyto_feature_cols_21]
enriched_features_val_21 = add_enriched_features(X_clinical_val_21[numeric_features], mol_features_val_21)

X_combined_val_21 = pd.concat([
    X_clinical_val_21, mol_features_val_21, cyto_features_val_21, enriched_features_val_21
], axis=1)

# ============================================================================
# Align features with V21.1 model
# ============================================================================
print(f"\n🔧 Aligning features with V21.1 model...")

# Remove duplicate columns (keep first occurrence)
X_combined_train_21 = X_combined_train_21.loc[:, ~X_combined_train_21.columns.duplicated()]
X_combined_val_21 = X_combined_val_21.loc[:, ~X_combined_val_21.columns.duplicated()]

# Add missing columns with zeros
for col in feature_columns_21_1:
    if col not in X_combined_train_21.columns:
        X_combined_train_21[col] = 0
    if col not in X_combined_val_21.columns:
        X_combined_val_21[col] = 0

# Reorder to match V21.1 exact order
X_combined_train_21 = X_combined_train_21[feature_columns_21_1]
X_combined_val_21 = X_combined_val_21[feature_columns_21_1]

print(f"✓ Features aligned: {X_combined_train_21.shape}")

# ============================================================================
# Preprocess using V21.1 imputer and scaler
# ============================================================================
X_train_cox_imputed = pd.DataFrame(
    imputer_cox_pretrained.transform(X_combined_train_21),
    index=X_combined_train_21.index,
    columns=X_combined_train_21.columns
)

X_val_cox_imputed = pd.DataFrame(
    imputer_cox_pretrained.transform(X_combined_val_21),
    index=X_combined_val_21.index,
    columns=X_combined_val_21.columns
)

X_train_cox_scaled = pd.DataFrame(
    scaler_cox_pretrained.transform(X_train_cox_imputed),
    index=X_train_cox_imputed.index,
    columns=X_train_cox_imputed.columns
)

X_val_cox_scaled = pd.DataFrame(
    scaler_cox_pretrained.transform(X_val_cox_imputed),
    index=X_val_cox_imputed.index,
    columns=X_val_cox_imputed.columns
)

# ============================================================================
# Generate predictions
# ============================================================================
y_pred_cox_val = cox_model_final.predict(X_val_cox_scaled.values)
c_index_cox = concordance_index_censored(
    y_val_ensemble['OS_STATUS'], y_val_ensemble['OS_YEARS'], y_pred_cox_val
)[0]

print(f"\n✓ CoxNet V21.1 predictions generated")
print(f"  C-index (ensemble val): {c_index_cox:.4f}")
print(f"  Features used: {np.sum(cox_model_final.coef_.flatten() != 0)}/{len(feature_columns_21_1)}")


MODEL 2: COXNET 21.1 - LOADING PRE-TRAINED MODEL

✓ Loaded pre-trained CoxNet V21.1
  Original C-index: 0.7493
  Features: 132
  Best params: l1_ratio=0.3, alpha=0.01

📝 Building features compatible with V21.1...

🔧 Aligning features with V21.1 model...
✓ Features aligned: (2855, 132)

✓ CoxNet V21.1 predictions generated
  C-index (ensemble val): 0.7482
  Features used: 95/132


## 6. MODEL 3: DeepSurv 20.1 (Full Training from V20.1)

In [19]:
print("\n" + "="*80)
print("MODEL 3: DEEPSURV 20.1 - FULL TRAINING")
print("="*80)

# ============================================================================
# Preprocess data for DeepSurv (use FULL V22.1 features, not V21.1 subset)
# ============================================================================

# DeepSurv needs its own preprocessing with ALL 159 features
imputer_nn = SimpleImputer(strategy='median')
X_train_nn_imputed = pd.DataFrame(
    imputer_nn.fit_transform(X_train_all),
    index=X_train_all.index,
    columns=X_train_all.columns
)

X_val_nn_imputed = pd.DataFrame(
    imputer_nn.transform(X_val_ensemble),
    index=X_val_ensemble.index,
    columns=X_val_ensemble.columns
)

scaler_nn = StandardScaler()
X_train_nn = pd.DataFrame(
    scaler_nn.fit_transform(X_train_nn_imputed),
    index=X_train_nn_imputed.index,
    columns=X_train_nn_imputed.columns
)

X_val_nn = pd.DataFrame(
    scaler_nn.transform(X_val_nn_imputed),
    index=X_val_nn_imputed.index,
    columns=X_val_nn_imputed.columns
)

y_train_nn = y_train_all.copy()
y_val_nn = y_val_ensemble.copy()

print(f"\n✓ Data prepared for DeepSurv")
print(f"  Features: {X_train_nn.shape[1]} (full V22.1 feature set)")
print(f"  Train samples: {X_train_nn.shape[0]}")
print(f"  Val samples: {X_val_nn.shape[0]}")

# ============================================================================
# PyTorch model and training
# ============================================================================

class SurvivalDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X.values)
        self.event = torch.FloatTensor(y['OS_STATUS'].astype(float))
        self.time = torch.FloatTensor(np.array(y['OS_YEARS'], dtype=np.float32))
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.event[idx], self.time[idx]

class ResidualBlock(nn.Module):
    def __init__(self, input_dim, output_dim, dropout=0.4):
        super(ResidualBlock, self).__init__()
        self.linear = nn.Linear(input_dim, output_dim)
        self.bn = nn.BatchNorm1d(output_dim)
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.skip = nn.Linear(input_dim, output_dim) if input_dim != output_dim else nn.Identity()
    
    def forward(self, x):
        identity = self.skip(x)
        out = self.linear(x)
        out = self.bn(out)
        out = self.activation(out)
        out = self.dropout(out)
        return out + identity

class ImprovedDeepSurv(nn.Module):
    def __init__(self, input_dim, hidden_layers=[256, 128, 64, 32], dropout=0.4):
        super(ImprovedDeepSurv, self).__init__()
        self.input_layer = ResidualBlock(input_dim, hidden_layers[0], dropout)
        self.hidden_layers = nn.ModuleList()
        for i in range(len(hidden_layers)-1):
            self.hidden_layers.append(ResidualBlock(hidden_layers[i], hidden_layers[i+1], dropout))
        self.output = nn.Linear(hidden_layers[-1], 1)
    
    def forward(self, x):
        x = self.input_layer(x)
        for layer in self.hidden_layers:
            x = layer(x)
        return self.output(x)

def cox_ph_loss(risk_scores, events, times):
    sorted_indices = torch.argsort(times, descending=True)
    risk_scores = risk_scores[sorted_indices].squeeze()
    events = events[sorted_indices]
    hazard_ratio = torch.exp(risk_scores)
    log_risk = torch.log(torch.cumsum(hazard_ratio, dim=0) + 1e-7)
    uncensored_likelihood = risk_scores - log_risk
    loss = -torch.sum(uncensored_likelihood * events) / (torch.sum(events) + 1e-7)
    return loss

def train_epoch(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0
    for X_batch, event_batch, time_batch in dataloader:
        X_batch = X_batch.to(device)
        event_batch = event_batch.to(device)
        time_batch = time_batch.to(device)
        optimizer.zero_grad()
        risk_scores = model(X_batch)
        loss = cox_ph_loss(risk_scores, event_batch, time_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)

def evaluate(model, dataloader, device):
    model.eval()
    all_risk_scores = []
    all_events = []
    all_times = []
    with torch.no_grad():
        for X_batch, event_batch, time_batch in dataloader:
            X_batch = X_batch.to(device)
            risk_scores = model(X_batch)
            all_risk_scores.extend(risk_scores.cpu().numpy())
            all_events.extend(event_batch.cpu().numpy())
            all_times.extend(time_batch.cpu().numpy())
    
    all_risk_scores = np.array(all_risk_scores).flatten()
    all_events = np.array(all_events).astype(bool)
    all_times = np.array(all_times)
    c_index = concordance_index_censored(all_events, all_times, all_risk_scores)[0]
    return c_index

# Create datasets
train_dataset = SurvivalDataset(X_train_nn, y_train_nn)
val_dataset = SurvivalDataset(X_val_nn, y_val_nn)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# Initialize model with correct input dimension
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_nn = ImprovedDeepSurv(input_dim=X_train_nn.shape[1], hidden_layers=[256, 128, 64, 32], dropout=0.4)
model_nn = model_nn.to(device)

# Training
optimizer = optim.Adam(model_nn.parameters(), lr=0.0005, weight_decay=0.0005)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=7)

best_val_c_index = 0
patience_counter = 0
patience = 20
num_epochs = 150

print(f"\nTraining on {device}...")
for epoch in range(num_epochs):
    train_loss = train_epoch(model_nn, train_loader, optimizer, device)
    val_c_index = evaluate(model_nn, val_loader, device)
    
    scheduler.step(train_loss)
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}/{num_epochs} | Loss: {train_loss:.4f} | Val C-index: {val_c_index:.4f}")
    
    if val_c_index > best_val_c_index:
        best_val_c_index = val_c_index
        best_model_state = model_nn.state_dict().copy()
        patience_counter = 0
    else:
        patience_counter += 1
    
    if patience_counter >= patience:
        print(f"Early stopping at epoch {epoch+1}")
        break

model_nn.load_state_dict(best_model_state)

# Get predictions on ensemble validation set
model_nn.eval()
with torch.no_grad():
    X_val_tensor = torch.FloatTensor(X_val_nn.values).to(device)
    y_pred_nn_val = model_nn(X_val_tensor).cpu().numpy().flatten()

c_index_nn = concordance_index_censored(
    y_val_ensemble['OS_STATUS'], y_val_ensemble['OS_YEARS'], y_pred_nn_val
)[0]

print(f"\n✓ DeepSurv trained on {X_train_nn.shape[0]} patients")
print(f"  Best val C-index: {best_val_c_index:.4f}")
print(f"  C-index (ensemble val): {c_index_nn:.4f}")


MODEL 3: DEEPSURV 20.1 - FULL TRAINING

✓ Data prepared for DeepSurv
  Features: 159 (full V22.1 feature set)
  Train samples: 2855
  Val samples: 318

Training on cpu...
Epoch  10/150 | Loss: 2.2964 | Val C-index: 0.7430
Epoch  20/150 | Loss: 2.1918 | Val C-index: 0.7475
Epoch  30/150 | Loss: 2.0797 | Val C-index: 0.7402
Early stopping at epoch 34

✓ DeepSurv trained on 2855 patients
  Best val C-index: 0.7569
  C-index (ensemble val): 0.7455


In [21]:
print("\n" + "="*80)
print("MODEL 3: DEEPSURV 20.1 - FULL TRAINING")
print("="*80)

# ============================================================================
# Preprocess data for DeepSurv (use FULL V22.1 features, not V21.1 subset)
# ============================================================================

imputer_nn = SimpleImputer(strategy='median')
X_train_nn_imputed = pd.DataFrame(
    imputer_nn.fit_transform(X_train_all),
    index=X_train_all.index,
    columns=X_train_all.columns
)

X_val_nn_imputed = pd.DataFrame(
    imputer_nn.transform(X_val_ensemble),
    index=X_val_ensemble.index,
    columns=X_val_ensemble.columns
)

scaler_nn = StandardScaler()
X_train_nn = pd.DataFrame(
    scaler_nn.fit_transform(X_train_nn_imputed),
    index=X_train_nn_imputed.index,
    columns=X_train_nn_imputed.columns
)

X_val_nn = pd.DataFrame(
    scaler_nn.transform(X_val_nn_imputed),
    index=X_val_nn_imputed.index,
    columns=X_val_nn_imputed.columns
)

y_train_nn = y_train_all.copy()
y_val_nn = y_val_ensemble.copy()

print(f"\n✓ Data prepared for DeepSurv")
print(f"  Features: {X_train_nn.shape[1]} (full V22.1 feature set)")
print(f"  Train samples: {X_train_nn.shape[0]}")
print(f"  Val samples: {X_val_nn.shape[0]}")

# ============================================================================
# PyTorch model and training
# ============================================================================

class SurvivalDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X.values)
        self.event = torch.FloatTensor(y['OS_STATUS'].astype(float))
        self.time = torch.FloatTensor(np.array(y['OS_YEARS'], dtype=np.float32))
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.event[idx], self.time[idx]

class ResidualBlock(nn.Module):
    def __init__(self, input_dim, output_dim, dropout=0.4):
        super(ResidualBlock, self).__init__()
        self.linear = nn.Linear(input_dim, output_dim)
        self.bn = nn.BatchNorm1d(output_dim)
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.skip = nn.Linear(input_dim, output_dim) if input_dim != output_dim else nn.Identity()
    
    def forward(self, x):
        identity = self.skip(x)
        out = self.linear(x)
        out = self.bn(out)
        out = self.activation(out)
        out = self.dropout(out)
        return out + identity

class ImprovedDeepSurv(nn.Module):
    def __init__(self, input_dim, hidden_layers=[256, 128, 64, 32], dropout=0.4):
        super(ImprovedDeepSurv, self).__init__()
        self.input_layer = ResidualBlock(input_dim, hidden_layers[0], dropout)
        self.hidden_layers = nn.ModuleList()
        for i in range(len(hidden_layers)-1):
            self.hidden_layers.append(ResidualBlock(hidden_layers[i], hidden_layers[i+1], dropout))
        self.output = nn.Linear(hidden_layers[-1], 1)
    
    def forward(self, x):
        x = self.input_layer(x)
        for layer in self.hidden_layers:
            x = layer(x)
        return self.output(x)

def cox_ph_loss(risk_scores, events, times):
    sorted_indices = torch.argsort(times, descending=True)
    risk_scores = risk_scores[sorted_indices].squeeze()
    events = events[sorted_indices]
    hazard_ratio = torch.exp(risk_scores)
    log_risk = torch.log(torch.cumsum(hazard_ratio, dim=0) + 1e-7)
    uncensored_likelihood = risk_scores - log_risk
    loss = -torch.sum(uncensored_likelihood * events) / (torch.sum(events) + 1e-7)
    return loss

def train_epoch(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0
    for X_batch, event_batch, time_batch in dataloader:
        X_batch = X_batch.to(device)
        event_batch = event_batch.to(device)
        time_batch = time_batch.to(device)
        optimizer.zero_grad()
        risk_scores = model(X_batch)
        loss = cox_ph_loss(risk_scores, event_batch, time_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)

def evaluate(model, dataloader, device):
    model.eval()
    all_risk_scores = []
    all_events = []
    all_times = []
    with torch.no_grad():
        for X_batch, event_batch, time_batch in dataloader:
            X_batch = X_batch.to(device)
            risk_scores = model(X_batch)
            all_risk_scores.extend(risk_scores.cpu().numpy())
            all_events.extend(event_batch.cpu().numpy())
            all_times.extend(time_batch.cpu().numpy())
    
    all_risk_scores = np.array(all_risk_scores).flatten()
    all_events = np.array(all_events).astype(bool)
    all_times = np.array(all_times)
    c_index = concordance_index_censored(all_events, all_times, all_risk_scores)[0]
    return c_index, all_risk_scores  # ⚠️ Retourner aussi les prédictions

# Create datasets
train_dataset = SurvivalDataset(X_train_nn, y_train_nn)
val_dataset = SurvivalDataset(X_val_nn, y_val_nn)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# Initialize model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_nn = ImprovedDeepSurv(input_dim=X_train_nn.shape[1], hidden_layers=[256, 128, 64, 32], dropout=0.4)
model_nn = model_nn.to(device)

# Training
optimizer = optim.Adam(model_nn.parameters(), lr=0.0005, weight_decay=0.0005)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=7)

best_val_c_index = 0
best_predictions = None  # ⚠️ Sauvegarder les prédictions du meilleur modèle
patience_counter = 0
patience = 20
num_epochs = 150

print(f"\nTraining on {device}...")
for epoch in range(num_epochs):
    train_loss = train_epoch(model_nn, train_loader, optimizer, device)
    val_c_index, val_predictions = evaluate(model_nn, val_loader, device)  # ⚠️ Récupérer les prédictions
    
    scheduler.step(train_loss)
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}/{num_epochs} | Loss: {train_loss:.4f} | Val C-index: {val_c_index:.4f}")
    
    if val_c_index > best_val_c_index:
        best_val_c_index = val_c_index
        best_model_state = model_nn.state_dict().copy()
        best_predictions = val_predictions.copy()  # ⚠️ Sauvegarder les prédictions
        patience_counter = 0
    else:
        patience_counter += 1
    
    if patience_counter >= patience:
        print(f"Early stopping at epoch {epoch+1}")
        break

model_nn.load_state_dict(best_model_state)

# ⚠️ Utiliser les prédictions sauvegardées du meilleur epoch
y_pred_nn_val = best_predictions
c_index_nn = best_val_c_index  # C'est le vrai meilleur C-index

print(f"\n✓ DeepSurv trained on {X_train_nn.shape[0]} patients")
print(f"  Best val C-index: {best_val_c_index:.4f}")
print(f"  Using best epoch predictions for ensemble")


MODEL 3: DEEPSURV 20.1 - FULL TRAINING

✓ Data prepared for DeepSurv
  Features: 159 (full V22.1 feature set)
  Train samples: 2855
  Val samples: 318

Training on cpu...
Epoch  10/150 | Loss: 2.2621 | Val C-index: 0.7462
Epoch  20/150 | Loss: 2.1644 | Val C-index: 0.7424
Epoch  30/150 | Loss: 2.0923 | Val C-index: 0.7453
Early stopping at epoch 32

✓ DeepSurv trained on 2855 patients
  Best val C-index: 0.7589
  Using best epoch predictions for ensemble


## 7. Optimize Ensemble Weights

In [32]:
print("\n" + "="*80)
print("OPTIMIZING ENSEMBLE WEIGHTS")
print("="*80)

def objective(trial):
    w_xgb = trial.suggest_float('w_xgb', 0.0, 1.0)
    w_cox = trial.suggest_float('w_cox', 0.0, 1.0 - w_xgb)
    w_nn = 1 - w_xgb - w_cox
    
    y_pred_ensemble = (
        w_xgb * y_pred_xgb_val + 
        w_cox * y_pred_cox_val + 
        w_nn * y_pred_nn_val
    )
    
    c_index = concordance_index_censored(
        y_val_ensemble['OS_STATUS'], y_val_ensemble['OS_YEARS'], y_pred_ensemble
    )[0]
    
    return c_index

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=2000, show_progress_bar=True)

best_w_xgb = study.best_params['w_xgb']
best_w_cox = study.best_params['w_cox']
best_w_nn = 1 - best_w_xgb - best_w_cox
best_c_index = study.best_value

best_weights = {
    'w_xgb': best_w_xgb,
    'w_cox': best_w_cox,
    'w_nn': best_w_nn
}   

print(f"\nBest weights:")
print(f"  XGBoost:  {best_w_xgb:.3f}")
print(f"  CoxNet:   {best_w_cox:.3f}")
print(f"  DeepSurv: {best_w_nn:.3f}")
print(f"\nEnsemble C-index: {best_c_index:.4f}")


OPTIMIZING ENSEMBLE WEIGHTS


  0%|          | 0/2000 [00:00<?, ?it/s]


Best weights:
  XGBoost:  0.213
  CoxNet:   0.026
  DeepSurv: 0.760

Ensemble C-index: 0.7631


## 8. Performance Comparison

In [26]:
print("\n" + "="*80)
print("PERFORMANCE COMPARISON")
print("="*80)

comparison = pd.DataFrame([
    {'Model': 'XGBoost 6.1', 'Val C-index': c_index_xgb, 'Weight': f'{best_w_xgb:.1%}'},
    {'Model': 'CoxNet 21.1', 'Val C-index': c_index_cox, 'Weight': f'{best_w_cox:.1%}'},
    {'Model': 'DeepSurv 20.1', 'Val C-index': c_index_nn, 'Weight': f'{best_w_nn:.1%}'},
    {'Model': 'V22.1 Ensemble', 'Val C-index': best_c_index, 'Weight': '100%'}
])

print("\n" + comparison.to_string(index=False))

improvement = best_c_index - max(c_index_xgb, c_index_cox, c_index_nn)
print(f"\n📈 Ensemble improvement: +{improvement:.4f}")
print(f"🎯 Target (0.75+): {'✅ ACHIEVED' if best_c_index >= 0.75 else '⏳ In progress'}")


PERFORMANCE COMPARISON

         Model  Val C-index Weight
   XGBoost 6.1     0.750368  21.3%
   CoxNet 21.1     0.748159   2.6%
 DeepSurv 20.1     0.758873  76.1%
V22.1 Ensemble     0.763181   100%

📈 Ensemble improvement: +0.0043
🎯 Target (0.75+): ✅ ACHIEVED


## 9. Final Predictions

In [33]:
print("\n" + "="*80)
print("FINAL PREDICTIONS ON TEST SET")
print("="*80)

# ============================================================================
# XGBoost predictions (uses full 159 features)
# ============================================================================
X_test_xgb = pd.DataFrame(
    imputer_xgb.transform(X_combined_test),
    index=X_combined_test.index,
    columns=X_combined_test.columns
)
dtest = xgb.DMatrix(X_test_xgb)
y_pred_xgb_test = xgb_model.predict(dtest)

print(f"✓ XGBoost predictions: {len(y_pred_xgb_test)}")

# ============================================================================
# CoxNet predictions (needs V21.1 compatible features: 132 features)
# ============================================================================
print(f"\n📝 Building V21.1-compatible features for test set...")

# Build V21.1 features for test set (same process as train/val)
X_clinical_test_21 = pd.DataFrame(index=X_combined_test.index)
for col in numeric_features:
    if col in X_combined_test.columns:
        X_clinical_test_21[col] = X_combined_test[col]

center_cols_test = [col for col in X_combined_test.columns if col.startswith('CENTER_')]
if len(center_cols_test) > 0:
    X_clinical_test_21 = pd.concat([X_clinical_test_21, X_combined_test[center_cols_test]], axis=1)

mol_features_test_21 = X_combined_test[mol_feature_cols]
cyto_features_test_21 = X_combined_test[cyto_feature_cols_21]
enriched_features_test_21 = add_enriched_features(X_clinical_test_21[numeric_features], mol_features_test_21)

X_combined_test_21 = pd.concat([
    X_clinical_test_21, mol_features_test_21, cyto_features_test_21, enriched_features_test_21
], axis=1)

# Remove duplicates and align with V21.1
X_combined_test_21 = X_combined_test_21.loc[:, ~X_combined_test_21.columns.duplicated()]

for col in feature_columns_21_1:
    if col not in X_combined_test_21.columns:
        X_combined_test_21[col] = 0

X_combined_test_21 = X_combined_test_21[feature_columns_21_1]

print(f"✓ V21.1 features built for test: {X_combined_test_21.shape}")

# Preprocess with V21.1 imputer and scaler
X_test_cox_imputed = pd.DataFrame(
    imputer_cox_pretrained.transform(X_combined_test_21),
    index=X_combined_test_21.index,
    columns=X_combined_test_21.columns
)

X_test_cox_scaled = pd.DataFrame(
    scaler_cox_pretrained.transform(X_test_cox_imputed),
    index=X_test_cox_imputed.index,
    columns=X_test_cox_imputed.columns
)

# CoxNet predictions
y_pred_cox_test = cox_model_final.predict(X_test_cox_scaled.values)

print(f"✓ CoxNet predictions: {len(y_pred_cox_test)}")

# ============================================================================
# DeepSurv predictions (uses full 159 features)
# ============================================================================
X_test_nn_imputed = pd.DataFrame(
    imputer_nn.transform(X_combined_test),
    index=X_combined_test.index,
    columns=X_combined_test.columns
)

X_test_nn_scaled = pd.DataFrame(
    scaler_nn.transform(X_test_nn_imputed),
    index=X_test_nn_imputed.index,
    columns=X_test_nn_imputed.columns
)

model_nn.eval()
with torch.no_grad():
    X_test_tensor = torch.FloatTensor(X_test_nn_scaled.values).to(device)
    y_pred_nn_test = model_nn(X_test_tensor).cpu().numpy().flatten()

print(f"✓ DeepSurv predictions: {len(y_pred_nn_test)}")

# ============================================================================
# Combine predictions with optimized weights
# ============================================================================
print(f"\n📊 Creating ensemble predictions...")

# Use the optimized weights from Section 7
ensemble_pred_test = (
    best_weights['w_xgb'] * y_pred_xgb_test +
    best_weights['w_cox'] * y_pred_cox_test +
    best_weights['w_nn'] * y_pred_nn_test
)

print(f"✓ Ensemble predictions created")
print(f"  XGBoost weight: {best_weights['w_xgb']:.4f}")
print(f"  CoxNet weight:  {best_weights['w_cox']:.4f}")
print(f"  DeepSurv weight: {best_weights['w_nn']:.4f}")

# ============================================================================
# Create submission file
# ============================================================================
submission = pd.DataFrame({
    'ID': X_combined_test.index,
    'OS_YEARS': ensemble_pred_test
})

submission_path = os.path.join(DATA_PATH, 'submission_v22.1_triple_ensemble_optimized.csv')
submission.to_csv(submission_path, index=False)

print(f"\n" + "="*80)
print("SUBMISSION FILE CREATED")
print("="*80)
print(f"✓ File: {submission_path}")
print(f"✓ Samples: {len(submission)}")
print(f"\nPreview:")
print(submission.head(10))
print(f"\nRisk score statistics:")
print(submission['OS_YEARS'].describe())


FINAL PREDICTIONS ON TEST SET
✓ XGBoost predictions: 1193

📝 Building V21.1-compatible features for test set...
✓ V21.1 features built for test: (1193, 132)
✓ CoxNet predictions: 1193
✓ DeepSurv predictions: 1193

📊 Creating ensemble predictions...
✓ Ensemble predictions created
  XGBoost weight: 0.2134
  CoxNet weight:  0.0265
  DeepSurv weight: 0.7602

SUBMISSION FILE CREATED
✓ File: C:/Users/guill/Desktop/Data Challenge QRT/Data-Challenge-Prediction-de-Survie\submission_v22.1_triple_ensemble_optimized.csv
✓ Samples: 1193

Preview:
      ID  OS_YEARS
0   KYW1  0.462238
1   KYW2  3.213037
2   KYW3 -0.254274
3   KYW4  0.213544
4   KYW5  1.708435
5   KYW6  2.550016
6   KYW7  1.319233
7   KYW8  0.745962
8   KYW9 -1.332598
9  KYW10 -1.754096

Risk score statistics:
count      1193.000000
mean       -109.234217
std        3800.432414
min     -131265.469020
25%          -0.854304
50%           0.389764
75%           1.659263
max          30.069040
Name: OS_YEARS, dtype: float64


## 10. Summary

In [34]:
print("\n" + "="*80)
print("SUMMARY - V22.1 FULL PIPELINE ENSEMBLE")
print("="*80)

print(f"\n🎯 Strategy:")
print(f"  XGBoost:  Direct training (like V6.1)")
print(f"  CoxNet:   Full CV for hyperparameters (like V21.1)")
print(f"  DeepSurv: Full training with early stopping (like V20.1)")

print(f"\n📊 Individual Model Performance:")
print(f"  XGBoost:  {c_index_xgb:.4f}")
print(f"  CoxNet:   {c_index_cox:.4f} (best: l1_ratio={best_l1_ratio}, alpha={best_alpha})")
print(f"  DeepSurv: {c_index_nn:.4f}")

print(f"\n🏆 Ensemble Performance:")
print(f"  Validation C-index: {best_c_index:.4f}")
print(f"  Weights: XGB {best_w_xgb:.1%} | Cox {best_w_cox:.1%} | NN {best_w_nn:.1%}")

print(f"\n✅ Output: {submission_path}")
print("="*80)


SUMMARY - V22.1 FULL PIPELINE ENSEMBLE

🎯 Strategy:
  XGBoost:  Direct training (like V6.1)
  CoxNet:   Full CV for hyperparameters (like V21.1)
  DeepSurv: Full training with early stopping (like V20.1)

📊 Individual Model Performance:
  XGBoost:  0.7504
  CoxNet:   0.7482 (best: l1_ratio=0.3, alpha=0.05)
  DeepSurv: 0.7589

🏆 Ensemble Performance:
  Validation C-index: 0.7631
  Weights: XGB 21.3% | Cox 2.6% | NN 76.0%

✅ Output: C:/Users/guill/Desktop/Data Challenge QRT/Data-Challenge-Prediction-de-Survie\submission_v22.1_triple_ensemble_optimized.csv
